# Prophet Tuning


This notebook is used to find the best hyperparameters for Prophet. Models are fit on the train data and validated on the calibration data.

Evaluation criteria:
- Metric: MAE (in MW) on the calibration split
- Each model is fit on the full train split and forecasts the whole calibration period, so no future data is used
- Train-internal walk-forward CV was not used: every fold had under 730 days of history, which is too little for Prophet's yearly seasonality and made the errors unstable
- Calibration is free to use here because Prophet has its own prediction intervals and does not need it for conformal calibration
- Seasonal naive (same day 364 days earlier) is the baseline to beat
- Test data is not used here

Best hyperparameters are chosen based on the lowest calibration MAE.

In [13]:
# 1. Load train (fit), calib (validation) and the holiday table. Test is never touched here.
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import itertools
import numpy as np, pandas as pd
from prophet import Prophet
from config import FEATURES_DATA_DIR

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)  # silence Prophet fit logs

def load(name):
    df = pd.read_parquet(FEATURES_DATA_DIR / name)[["ds", "y"]]
    df["ds"] = pd.to_datetime(df["ds"])
    return df

train, calib = load("series_train.parquet"), load("series_calib.parquet")
holidays_df = pd.read_parquet(FEATURES_DATA_DIR / "prophet_holidays.parquet")  # needs columns: holiday, ds
print(len(train), len(calib))

852 366


In [20]:
# 2. Settings kept fixed vs settings searched
FIXED = {"weekly_seasonality": True, "yearly_seasonality": True,
         "daily_seasonality": False, "interval_width": 0.90}

grid = {
    "changepoint_prior_scale" : [0.0001, 0.0005, 0.001, 0.002],  # trend flexibility (lower = stiffer trend)
    "seasonality_prior_scale": [1, 10],                      # seasonality flexibility
    "holidays_prior_scale": [1, 10],                         # holiday effect strength
    "seasonality_mode": ["additive", "multiplicative"],
}

In [21]:
# 3. Fit on the full train (852 days, enough history for yearly seasonality),
# score MAE (MW) on calib. Prophet has its own intervals, so calib is free to use for validation.
results = []
for vals in itertools.product(*grid.values()):
    params = dict(zip(grid.keys(), vals))
    m = Prophet(**FIXED, **params, holidays=holidays_df).fit(train)
    yhat = m.predict(calib[["ds"]])["yhat"].values
    results.append({"params": params, "val_mae": np.mean(np.abs(calib["y"].values - yhat))})

results = sorted(results, key=lambda r: r["val_mae"])
pd.DataFrame(results).head(10)

,params,val_mae
0,"{'changepoint_prior_scale': 0.0005, 'seasonali...",6950.005344
1,"{'changepoint_prior_scale': 0.0005, 'seasonali...",6950.011285
2,"{'changepoint_prior_scale': 0.0005, 'seasonali...",6950.171581
3,"{'changepoint_prior_scale': 0.0005, 'seasonali...",6950.173633
4,"{'changepoint_prior_scale': 0.0005, 'seasonali...",7073.892899
5,"{'changepoint_prior_scale': 0.0005, 'seasonali...",7074.875667
6,"{'changepoint_prior_scale': 0.0005, 'seasonali...",7246.788949
7,"{'changepoint_prior_scale': 0.0005, 'seasonali...",7307.789271
8,"{'changepoint_prior_scale': 0.0001, 'seasonali...",7757.889879
9,"{'changepoint_prior_scale': 0.0001, 'seasonali...",7757.978925


In [22]:
# 4. Best params. If one sits on the edge of the grid, widen the grid in that direction and rerun.
# Copy this dict into PROPHET_PARAMS in config.py.
best = results[0]["params"]
print(best, "| calib MAE:", round(results[0]["val_mae"], 1))

{'changepoint_prior_scale': 0.0005, 'seasonality_prior_scale': 1, 'holidays_prior_scale': 10, 'seasonality_mode': 'additive'} | calib MAE: 6950.0


In [23]:
# 5. Baseline to beat at a long horizon: seasonal naive (same day 364 days earlier, same weekday)
series = pd.read_parquet(FEATURES_DATA_DIR / "series.parquet")
series["ds"] = pd.to_datetime(series["ds"])
s = series.sort_values("ds").set_index("ds")["y"]
sn = s.shift(364).reindex(calib["ds"]).values
print("seasonal-naive calib MAE:", round(np.nanmean(np.abs(calib["y"].values - sn))))

seasonal-naive calib MAE: 15441


In [24]:
import json
print(json.dumps(best, indent=2))

{
  "changepoint_prior_scale": 0.0005,
  "seasonality_prior_scale": 1,
  "holidays_prior_scale": 10,
  "seasonality_mode": "additive"
}
